> **Cópia pública saneada.** Os dados de entrada não acompanham este repositório. Leia `docs/reprodutibilidade.md` e `docs/privacidade_e_dados.md` antes da execução. Notebooks de coleta dependem de rede; notebooks de tratamento escrevem somente em `data/`, que é ignorada pelo Git.

# Armazenamento dos dados

Este notebook organiza os arquivos brutos baixados do portal de dados abertos do BNDES.

A etapa de armazenamento tem três objetivos principais:

1. verificar quais arquivos CSV e PDF estão disponíveis nas pastas do projeto;
2. registrar informações básicas dos arquivos, como nome, formato e tamanho;
3. preparar os dados para leitura e análise posterior, sem alterar os arquivos originais.

Os arquivos brutos permanecem em `data/raw`. As versões organizadas ou convertidas serão salvas em `data/interim`.

In [ ]:
from pathlib import Path
from datetime import datetime

import pandas as pd

## 1. Localização das pastas do projeto

Neste bloco, definimos os caminhos principais usados no notebook.

A pasta `data/raw` contém os arquivos brutos baixados do BNDES. A pasta `data/interim` será usada para armazenar versões intermediárias, mais organizadas e adequadas para análise.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_CSV_DIR = RAW_DIR / "csv"
RAW_PDF_DIR = RAW_DIR / "pdf"

INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "results" / "tables"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## 2. Inventário dos arquivos brutos

Neste bloco, criamos um inventário dos arquivos que foram baixados para a pasta `data/raw`.

O inventário registra o nome do arquivo, o formato, o caminho local, o tamanho em megabytes e a data de modificação. Essa etapa é importante para documentar o que está disponível antes de iniciar qualquer leitura ou transformação dos dados.

In [ ]:
arquivos_raw = []

for pasta in [RAW_CSV_DIR, RAW_PDF_DIR]:
    for arquivo in sorted(pasta.glob("*")):
        if arquivo.is_file():
            arquivos_raw.append({
                "nome_arquivo": arquivo.name,
                "formato": arquivo.suffix.lower().replace(".", ""),
                "pasta": arquivo.parent.name,
                "caminho": arquivo,
                "tamanho_mb": arquivo.stat().st_size / (1024 * 1024),
                "data_modificacao": datetime.fromtimestamp(
                    arquivo.stat().st_mtime
                ),
            })

inventario_raw = pd.DataFrame(arquivos_raw)

inventario_raw

## 3. Resumo dos arquivos por formato

Neste bloco, resumimos o inventário dos arquivos brutos por formato.

Esse resumo permite verificar quantos arquivos CSV e PDF estão disponíveis e qual é o tamanho total aproximado de cada grupo. Isso é especialmente importante porque alguns CSVs do BNDES são grandes e exigem cuidado na leitura.

In [ ]:
resumo_raw = (
    inventario_raw
    .groupby(["formato", "pasta"], dropna=False)
    .agg(
        quantidade_arquivos=("nome_arquivo", "count"),
        tamanho_total_mb=("tamanho_mb", "sum"),
        tamanho_medio_mb=("tamanho_mb", "mean"),
    )
    .reset_index()
    .sort_values(["formato", "pasta"])
)

resumo_raw

## 4. Separação dos arquivos CSV para armazenamento

Neste bloco, separamos apenas os arquivos CSV do inventário bruto.

Os CSVs são as bases de dados que serão lidas e transformadas para um formato intermediário mais eficiente. Os PDFs permanecem como documentação metodológica e não entram na etapa de armazenamento tabular.

In [ ]:
inventario_csv = (
    inventario_raw
    .query("formato == 'csv'")
    .copy()
    .sort_values("nome_arquivo")
    .reset_index(drop=True)
)

inventario_csv

## 5. Identificação do separador e encoding dos CSVs

Antes de ler as bases completas, vamos inspecionar uma pequena amostra de cada arquivo CSV.

O objetivo é verificar se os arquivos usam separador `;`, encoding compatível e quais colunas aparecem em cada base. Essa etapa evita erros de leitura, especialmente em bases grandes.

In [ ]:
encodings_teste = ["utf-8", "windows-1252", "latin1"]

amostras_csv = []

for _, linha in inventario_csv.iterrows():
    caminho = linha["caminho"]
    leitura_ok = False

    for encoding in encodings_teste:
        try:
            amostra = pd.read_csv(
                caminho,
                sep=";",
                encoding=encoding,
                nrows=5,
                low_memory=False
            )

            amostras_csv.append({
                "nome_arquivo": linha["nome_arquivo"],
                "encoding": encoding,
                "quantidade_colunas": amostra.shape[1],
                "colunas": list(amostra.columns),
                "status": "ok",
            })

            leitura_ok = True
            break

        except UnicodeDecodeError:
            continue

    if not leitura_ok:
        amostras_csv.append({
            "nome_arquivo": linha["nome_arquivo"],
            "encoding": None,
            "quantidade_colunas": None,
            "colunas": None,
            "status": "erro_encoding",
        })

amostras_csv = pd.DataFrame(amostras_csv)

amostras_csv

## 6. Salvamento do diagnóstico de leitura dos CSVs

Neste bloco, salvamos o diagnóstico inicial dos arquivos CSV.

Esse arquivo registra qual encoding funcionou para cada base, quantas colunas foram identificadas e se a leitura da amostra foi bem-sucedida.

Essa informação será usada nas próximas etapas para ler cada arquivo corretamente.

In [ ]:
arquivo_diagnostico_csv = OUTPUT_TABLES_DIR / "diagnostico_leitura_csv.xlsx"

amostras_csv.to_excel(
    arquivo_diagnostico_csv,
    index=False,
    sheet_name="Diagnostico_CSV"
)

arquivo_diagnostico_csv

## 7. Criação do mapa de encoding dos arquivos CSV

Neste bloco, transformamos o diagnóstico de leitura em um mapa de encoding.

Esse mapa associa cada arquivo CSV ao encoding que funcionou na leitura da amostra. Ele será usado para ler os arquivos completos de forma mais segura nas próximas etapas.

In [ ]:
mapa_encoding_csv = dict(
    zip(
        amostras_csv["nome_arquivo"],
        amostras_csv["encoding"]
    )
)

mapa_encoding_csv

## 8. Definição dos nomes dos arquivos intermediários

Neste bloco, definimos o nome de saída de cada arquivo CSV convertido para o formato Parquet.

O formato Parquet é mais eficiente para armazenamento e leitura de bases grandes, pois ocupa menos espaço e preserva melhor os tipos de dados. Os arquivos convertidos serão salvos em `data/interim`.

Os arquivos CSV originais permanecem inalterados em `data/raw/csv`.

In [ ]:
inventario_csv["nome_parquet"] = (
    inventario_csv["nome_arquivo"]
    .str.replace(".csv", ".parquet", regex=False)
)

inventario_csv["caminho_parquet"] = inventario_csv["nome_parquet"].apply(
    lambda nome: INTERIM_DIR / nome
)

inventario_csv[
    ["nome_arquivo", "encoding", "nome_parquet", "caminho_parquet"]
] if "encoding" in inventario_csv.columns else inventario_csv[
    ["nome_arquivo", "nome_parquet", "caminho_parquet"]
]